# `unwrap_trajectory()`

The grid utility `nematics3d.unwrap_trajectory()` converts a trajectory stored in periodic-box coordinates into one continuous path. When consecutive samples cross a periodic boundary, their wrapped displacement is replaced by its nearest periodic image.

Three facts are important from the beginning:

- `box_size_periodic` may be one shared box length or three values for the $x$, $y$, and $z$ axes. `np.inf` marks a non-periodic axis.
- The physical displacement between consecutive samples must be smaller than half the corresponding box length; otherwise the periodic coordinates do not determine a unique path.
- The continuous result may lie outside the principal box. `is_start_in_box=True` applies one whole-box translation to place a selected reference point inside it.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell imports `NumPy` and `Nematics3D`.


In [1]:
import numpy as np
import nematics3d as n3d


## Minimal example: cross one periodic boundary

The third sample wraps from the high-$x$ side of a box of length 10 to the low-$x$ side. Its stored coordinate is `0.5`, while the continuous path reaches `10.5`.


In [2]:
wrapped = np.array(
    [[8.0, 2.0, 0.0], [9.5, 2.0, 0.0], [0.5, 2.0, 0.0], [2.0, 2.0, 0.0]]
)
unwrapped = n3d.unwrap_trajectory(wrapped, box_size_periodic=10.0)
print(unwrapped)


[[ 8.   2.   0. ]
 [ 9.5  2.   0. ]
 [10.5  2.   0. ]
 [12.   2.   0. ]]


## Inputs and outputs

The public signature is:

```python
unwrap_trajectory(
    points,
    box_size_periodic=np.inf,
    *,
    is_start_in_box=False,
    ref_index=0,
    is_reverse=False,
)
```

### Accepted trajectory representations

`points` must contain real, finite three-dimensional coordinates in one of these forms:

| Representation | Input shape | Normalized shape | Meaning |
| --- | --- | --- | --- |
| Single point | `(3,)` | `(1, 3)` | One trajectory sample |
| Point sequence | `(N, 3)` | `(N, 3)` | `N` samples in trajectory order |
| Empty sequence | `(0, 3)` or `[]` | `(0, 3)` | No samples |

Lists, tuples, and `NumPy` arrays are accepted. Boolean, string, complex, `NaN`, and infinite coordinates are rejected. Shapes other than `(3,)` or `(N, 3)` are rejected. Empty input is valid only when no principal-box reference point is requested.

### Accepted periodic-box representations

`box_size_periodic` assigns a box length to each coordinate axis:

| Representation | Example | Meaning |
| --- | --- | --- |
| One scalar | `10.0` | All three axes have period 10 |
| Three values | `[10.0, 20.0, 30.0]` | Separate $x$, $y$, and $z$ periods |
| Mixed finite and infinite values | `[10.0, 20.0, np.inf]` | $x$ and $y$ are periodic; $z$ is not |
| Positive infinity | `np.inf` | No axis is periodic |

Every finite period must be positive. Positive infinity is the only non-periodic marker. Boolean values, `NaN`, negative infinity, zero, negative values, complex values, strings, and shapes other than a scalar or `(3,)` are rejected.

### Options

| Argument | Meaning |
| --- | --- |
| `is_start_in_box=False` | Preserve the periodic image containing the selected reconstruction anchor |
| `is_start_in_box=True` | Translate the complete result by whole box lengths until the point at `ref_index` lies in the principal box |
| `ref_index=0` | Select the reference point for the optional final translation; negative indices follow normal `NumPy` indexing |
| `is_reverse=False` | Reconstruct outward from the first input point |
| `is_reverse=True` | Reconstruct outward from the final input point, then restore the original point ordering |

`ref_index` is validated only when `is_start_in_box=True`, because it otherwise has no effect. Boolean options accept booleans and numeric zero or one; other values are rejected.

### Returned trajectory

The function returns one floating-point `NumPy` array:

| Property | Result |
| --- | --- |
| Shape | `(N, 3)`, including `(1, 3)` for one input point and `(0, 3)` for empty input |
| Point order | The same order as the input, including when `is_reverse=True` |
| Continuity | Consecutive displacements use their minimum periodic images |
| Global position | Anchored at the first or final stored point, with an optional whole-box translation |
| Ownership | Independent of the input; modifying the result does not modify `points` |


## Examples


### Mixed periodic axes and endpoint choice

Only finite box-size entries are corrected. Reverse mode is useful when the final stored point is the endpoint whose periodic image should remain fixed. Forward and reverse results can differ by whole box lengths while describing the same local path.


In [3]:
mixed = np.array([[9.0, 1.0, 0.0], [1.0, 19.0, 2.0], [3.0, 17.0, 4.0]])
print(n3d.unwrap_trajectory(mixed, [10.0, 20.0, np.inf]))

endpoint_example = np.array(
    [[1.0, 0.0, 0.0], [9.0, 0.0, 0.0], [7.0, 0.0, 0.0]]
)
print("first anchored:\n", n3d.unwrap_trajectory(endpoint_example, 10.0))
print("last anchored:\n", n3d.unwrap_trajectory(endpoint_example, 10.0, is_reverse=True))


[[ 9.  1.  0.]
 [11. -1.  2.]
 [13. -3.  4.]]
first anchored:
 [[ 1.  0.  0.]
 [-1.  0.  0.]
 [-3.  0.  0.]]
last anchored:
 [[11.  0.  0.]
 [ 9.  0.  0.]
 [ 7.  0.  0.]]


### Placing a reference point in the principal box

The optional final shift adds the same integer number of box lengths to every point along each periodic axis. It changes the global periodic image without changing trajectory shape or relative displacement.


In [4]:
outside_box = np.array(
    [[19.0, 0.0, 0.0], [1.0, 0.0, 0.0], [3.0, 0.0, 0.0]]
)
placed = n3d.unwrap_trajectory(
    outside_box, 10.0, is_start_in_box=True, ref_index=1
)
print(placed)


[[-1.  0.  0.]
 [ 1.  0.  0.]
 [ 3.  0.  0.]]


## Details

### Minimum-image reconstruction

For each periodic axis of length $L$, the function computes consecutive displacements and subtracts the nearest integer multiple of $L$. The corrected displacements are then cumulatively summed from the selected endpoint. `NumPy` writes this cumulative sum directly into the output array, avoiding an additional trajectory-sized allocation.

### Reverse anchoring and principal-box placement

Reverse mode temporarily processes the samples from last to first, thereby retaining the final stored point as the reconstruction anchor, and then restores the original order. Principal-box placement is a separate final operation: it adds the same whole-box displacement to every point along each periodic axis, preserving all relative positions.


## Possible issues

### Sampling too sparsely

Periodic coordinates do not reveal how many times a path crossed the box between samples. If motion can exceed half a box length per sample, save frames more frequently or retain separate image counters.

### Exactly half a box length

A displacement of exactly $L/2$ has two equally short periodic images. The implementation follows `numpy.round()` and its tie-to-even behavior, but the physical direction cannot be inferred from periodic coordinates alone.

### Point order matters

Adjacent rows are interpreted as consecutive samples. An unordered point cloud cannot be repaired by trajectory unwrapping.


## Logging and diagnostic information

Set `log_level=logging.DEBUG` to report the point count, normalized `box_size_periodic`, endpoint choice, and optional principal-box shift. For example, `[10.0, inf, inf]` means that only the $x$ axis is periodic. Complete trajectory arrays are intentionally not logged.


In [5]:
import logging

debug_result = n3d.unwrap_trajectory(
    wrapped,
    [10.0, np.inf, np.inf],
    is_start_in_box=True,
    log_level=logging.DEBUG,
)


[DEBUG]
    <unwrap_trajectory> 
    Unwrapping 4 point(s); periodic axes=[0], is_reverse=False, is_start_in_box=True.
[DEBUG]
    <unwrap_trajectory> 
    Shifting the unwrapped trajectory so point 0 lies in the principal periodic box.


## Where `unwrap_trajectory()` is used

`Nematics3D` uses this utility in [`defect_classify_into_lines()`](../../../src/nematics3d/analysis/disclination/classification.py) and [`DisclinationLine`](../../../src/nematics3d/classes/disclination_line.py) to turn periodic defect indices into continuous line coordinates.


## Useful Links

### Referenced in this tutorial

- [`unwrap_trajectory()` source](../../../src/nematics3d/grid/periodic.py) — implements minimum-image reconstruction and optional principal-box placement.
- [`as_points()` source](../../../src/nematics3d/datatypes/points.py) — defines trajectory shape, coordinate, finite-value, and ownership validation.
- [`as_box_size_periodic()` source](../../../src/nematics3d/datatypes/box_size_periodic.py) — defines scalar and per-axis periodic-box representations.
- [`defect_classify_into_lines()` source](../../../src/nematics3d/analysis/disclination/classification.py) — unwraps classified periodic defect trajectories.
- [`DisclinationLine` source](../../../src/nematics3d/classes/disclination_line.py) — unwraps periodic lattice indices for line geometry.

### Going deeper

- [Defect detection](../../analysis/disclination/defect_detect.ipynb) — finds defect indices that can later be classified into trajectories.
- [QFieldObject defect detection](../../classes/QFieldObject/act_defect_detect.ipynb) — runs defect detection through the high-level $Q$-field object.
- [QFieldObject line classification](../../classes/QFieldObject/act_lines_classify.ipynb) — classifies detected defects into line objects.
